In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

In [3]:
# 1. 브라우저 설정
options = Options()
# options.add_argument("--headless") # 창을 띄우지 않으려면 주석 해제
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def crawl_yanolja_reviews(url):
    driver.get(url)
    time.sleep(3)

    # 2. 모든 리뷰 로드를 위한 무한 스크롤
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # 3. 데이터 수집
    review_data = []
    # 최상위 컨테이너로 각 리뷰 묶음 탐색
    items = driver.find_elements(By.CLASS_NAME, "review-item-container")

    for item in items:
        try:
            # [작성일자] css-1ivchjf 하위의 css-6lreu3
            date = item.find_element(By.CSS_SELECTOR, ".css-1ivchjf .css-6lreu3").text

            # [아이디] css-iltqar 하위의 첫 번째 p(css-6lreu3) 안의 span
            user_id = item.find_element(By.CSS_SELECTOR, ".css-iltqar p.css-6lreu3 span").text

            # [객실명] css-iltqar 하위의 두 번째 p(css-6lreu3)
            # 동일 클래스가 나열되어 있으므로 elements로 받아 두 번째(index 1) 선택
            room_info_elements = item.find_elements(By.CSS_SELECTOR, ".css-iltqar .css-6lreu3")
            room_type = room_info_elements[1].text if len(room_info_elements) > 1 else "정보없음"

            # [리뷰내용] 클래스명 두 개가 붙어있는 구조
            content = item.find_element(By.CLASS_NAME, "content-text").text

            review_data.append({
                "작성일자": date,
                "아이디": user_id,
                "객실명": room_type,
                "리뷰내용": content
            })
        except Exception as e:
            print(f"항목 추출 중 건너뜀: {e}")
            continue

    return review_data
# 실행 및 저장
try:
    url = "https://nol.yanolja.com/reviews/domestic/10046470?sort=HOST_CHOICE"
    results = crawl_yanolja_reviews(url)

    # 4. 엑셀 저장 (Pandas 활용)
    df = pd.DataFrame(results)
    df.to_excel("nol_reviews.xlsx", index=False)
    print(f"총 {len(df)}건의 리뷰가 'nol_reviews.xlsx'로 저장되었습니다.")

finally:
    driver.quit()

총 511건의 리뷰가 'nol_reviews.xlsx'로 저장되었습니다.
